# 06 · The decision pipeline — production, cell by cell

**Purpose.** Walk the REAL production decision path
(`scripts/wca_betrecs.py` — the code that builds the shipped
recommendation feed) stage by stage, showing per stage: input schema,
the transform, parameters, the intermediate frame, rows accepted vs
rejected with reasons, and the output schema. Then trace individual
candidates end-to-end and prove parity against the shipped feed.

Stages:
feeds → pool resolution (combined £3,000±PnL bankroll, ¼-Kelly) →
match singles (blend vs de-vigged consensus) → event props →
advancement futures (PM, fee-adjusted) → governance caps →
actionable/withheld split.

**This notebook only READS.** It never fires orders, never writes any
feed, never touches `pm_parked`.

In [1]:
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
JB = next(q for q in [_here, *_here.parents] if (q / "lib" / "bootstrap.py").exists())
sys.path.insert(0, str(JB))

import json
import pandas as pd
import polars as pl
import lib.bootstrap as bt
import lib.config as cfg
import lib.storage as st
import lib.pipeline as pp

manifest = bt.run_manifest("06_decision_pipeline")
p = cfg.load_params()

## Stage 0a · Input feeds (exactly what production reads) + freshness
Staleness gates downstream depend on these ages — production withholds on
stale feeds rather than recommending from them.

In [2]:
feeds = pp.load_feeds()
pd.DataFrame([{
    "feed": k, "path": v["path"].replace(str(bt.REPO_ROOT) + "/", ""),
    "age_h": round(v["age_secs"] / 3600, 1) if v["age_secs"] is not None else None,
    "items": len((v["data"] or {}).get("predictions")
                 or (v["data"] or {}).get("markets")
                 or (v["data"] or {}).get("recommendations")
                 or (v["data"] or {}).get("promotions") or []),
} for k, v in feeds.items()])

,feed,path,age_h,items
0,model_predictions,data/model_predictions.json,1.7,0
1,advancement,site/advancement_data.json,1.7,0
2,promos,site/promos_data.json,2.9,0
3,scores_markets,site/scores_markets.json,0.7,0
4,prop_calibration,data/prop_calibration.json,NaN,0
5,arb,site/arb_data.json,49.1,0
6,exposure,site/exposure_dashboard.json,NaN,0
7,bet_recs_shipped,site/bet_recs.json,1.5,0


## Stage 0b · Bankroll pools — production resolution
ONE combined bankroll (£3,000 ± total realised P&L, GBP+PM at $1.33/£),
¼-Kelly, with static fail-closed caps. The dev-box ledger is STALE — the
caveat rides along in the output rather than being hidden.

In [3]:
pools = pp.resolve_pools()
pd.Series({
    "sportsbook pool £": pools["sportsbook"]["bankroll"],
    "sportsbook kelly": pools["sportsbook"]["kelly_fraction"],
    "sportsbook max stake £": pools["sportsbook"]["max_stake"],
    "pm pool $": pools["pm"]["bankroll"],
    "pm kelly": pools["pm"]["kelly_fraction"],
    "pm max stake $": pools["pm"]["max_stake"],
    "pm realised pnl $ (stale dev ledger)": pools["pm_realised_pnl_usd"],
    "caveat": pools["ledger_caveat"],
}).to_frame("value")

,value
sportsbook pool £,3134.784372
sportsbook kelly,0.25
sportsbook max stake £,156.74
pm pool $,4034.49
pm kelly,0.25
pm max stake $,161.38
pm realised pnl $ (stale dev ledger),44.487
caveat,dev-box wca.db is a STALE copy; canonical ledg...


## Stages 1–3 · Run the production builders
`build_match_singles` (model blend vs de-vigged consensus, 2pp edge gate,
staleness + exposure guards, promo tags) · `build_event_props`
(calibrated corners/cards — withheld unless real fresh prices exist) ·
`build_advancement_futures` (PM advancement vs sim, fee-adjusted edges,
¼-Kelly, PM-blind + 6h staleness guards).

In [4]:
stages = pp.run_stages(feeds, pools)
funnel = pp.funnel(stages)
funnel.to_pandas()

,stage,actionable,withheld,top_reject_reasons
0,singles,3,9,"{""unspecified"": 9}"
1,props,0,9,"{""unspecified"": 9}"
2,advancement,14,0,{}


### Intermediate frames — schema, rows, nulls (transparency contract)

In [5]:
frames = pp.stage_frames(stages)
for name, df in frames.items():
    if df.height:
        prof = st.profile_frame(df, name)
        print(f"— {name}: {prof.attrs['rows']} rows × {df.width} cols")
for name in ("singles_actionable", "advancement_actionable"):
    if frames[name].height:
        display(frames[name].head(6).to_pandas())

— singles_actionable: 3 rows × 26 cols
— singles_withheld: 9 rows × 16 cols
— props_withheld: 9 rows × 8 cols
— advancement_actionable: 14 rows × 25 cols


,id,fixture,kickoff,group,market,market_kind,market_label,selection,team,venue,...,stake,action_label,current_exposure,proposed_risk,promo,promo_status,ages,stale,stale_reason,tags
0,brazil_vs_norway_home_1x2,Brazil vs Norway,2026-07-05 20:00:00+00:00,,1X2,result_90,90-min 1X2 (+ stoppage),home,Brazil,smarkets,...,82.80,ADD,"{""fixture_open"": false}","{""stake"": 82.8, ""max_loss"": 82.8}",None,None,"{""model_secs"": 5972, ""price_secs"": 5972, ""expo...",False,None,"[""model"", ""1X2"", ""devig_price""]"
1,united_states_vs_belgium_away_1x2,United States vs Belgium,2026-07-07 00:00:00+00:00,,1X2,result_90,90-min 1X2 (+ stoppage),away,Belgium,smarkets,...,42.95,ADD,"{""fixture_open"": false}","{""stake"": 42.95, ""max_loss"": 42.95}",None,None,"{""model_secs"": 5972, ""price_secs"": 5972, ""expo...",False,None,"[""model"", ""1X2"", ""devig_price""]"
2,mexico_vs_england_away_1x2,Mexico vs England,2026-07-06 00:00:00+00:00,,1X2,result_90,90-min 1X2 (+ stoppage),away,England,smarkets,...,26.97,ADD,"{""fixture_open"": false}","{""stake"": 26.97, ""max_loss"": 26.97}",None,None,"{""model_secs"": 5972, ""price_secs"": 5972, ""expo...",False,None,"[""model"", ""1X2"", ""devig_price""]"


,id,team,group,stage,market,selection,venue,currency,model_prob,pm_price,...,action_label,ages,stale,stale_reason,tags,market_kind,market_label,opponent,match_round,match_1x2
0,belgium_qf_pm,Belgium,G,QF,advancement,reach_QF,polymarket,USD,0.6858,0.5000,...,ADD,"{""model_secs"": 6055, ""price_secs"": 6055}",False,None,"[""model"", ""advancement"", ""polymarket""]",advancement,Advance · incl. ET+pens,United States,Round of 16,"{""team_win"": 0.499, ""draw"": 0.269, ""opp_win"": ..."
1,brazil_sf_pm,Brazil,C,SF,advancement,reach_SF,polymarket,USD,0.4920,0.3550,...,ADD,"{""model_secs"": 6055, ""price_secs"": 6055}",False,None,"[""model"", ""advancement"", ""polymarket""]",advancement,Advance · incl. ET+pens,Norway,Round of 16,"{""team_win"": 0.597, ""draw"": 0.239, ""opp_win"": ..."
2,england_qf_pm,England,L,QF,advancement,reach_QF,polymarket,USD,0.6725,0.5350,...,ADD,"{""model_secs"": 6055, ""price_secs"": 6055}",False,None,"[""model"", ""advancement"", ""polymarket""]",advancement,Advance · incl. ET+pens,Mexico,Round of 16,"{""team_win"": 0.468, ""draw"": 0.287, ""opp_win"": ..."
3,brazil_final_pm,Brazil,C,Final,advancement,reach_Final,polymarket,USD,0.3024,0.1800,...,ADD,"{""model_secs"": 6055, ""price_secs"": 6055}",False,None,"[""model"", ""advancement"", ""polymarket""]",advancement,Advance · incl. ET+pens,Norway,Round of 16,"{""team_win"": 0.597, ""draw"": 0.239, ""opp_win"": ..."
4,brazil_win_pm,Brazil,C,win,advancement,reach_win,polymarket,USD,0.1744,0.0625,...,ADD,"{""model_secs"": 6055, ""price_secs"": 6055}",False,None,"[""model"", ""advancement"", ""polymarket""]",advancement,Advance · incl. ET+pens,Norway,Round of 16,"{""team_win"": 0.597, ""draw"": 0.239, ""opp_win"": ..."
5,brazil_qf_pm,Brazil,C,QF,advancement,reach_QF,polymarket,USD,0.7796,0.6650,...,ADD,"{""model_secs"": 6055, ""price_secs"": 6055}",False,None,"[""model"", ""advancement"", ""polymarket""]",advancement,Advance · incl. ET+pens,Norway,Round of 16,"{""team_win"": 0.597, ""draw"": 0.239, ""opp_win"": ..."


### Why rows were rejected — the withheld tables (full reasons)

In [6]:
for name in ("singles_withheld", "props_withheld", "advancement_withheld"):
    df = frames[name]
    if df.height:
        cols = [c for c in ("fixture", "selection", "market", "team", "stage",
                            "reason", "action") if c in df.columns]
        print(f"— {name} ({df.height} rows)")
        display(df.select(cols).head(8).to_pandas())

— singles_withheld (9 rows)


,fixture,selection,market,team
0,Canada vs Morocco,home,1X2,Canada
1,Paraguay vs France,home,1X2,Paraguay
2,Paraguay vs France,draw,1X2,Draw
3,Brazil vs Norway,away,1X2,Norway
4,Portugal vs Spain,home,1X2,Portugal
5,Argentina vs Egypt,draw,1X2,Draw
6,Argentina vs Egypt,away,1X2,Egypt
7,Argentina vs Egypt,draw,1X2,Draw


— props_withheld (9 rows)


,fixture,selection,market
0,Qatar vs Switzerland,corners over 8.5,corners
1,Qatar vs Switzerland,cards over 2.5,cards
2,Brazil vs Morocco,corners over 8.5,corners
3,Brazil vs Morocco,cards over 2.5,cards
4,Haiti vs Scotland,corners over 8.5,corners
5,Haiti vs Scotland,cards over 2.5,cards
6,Australia vs Turkey,corners over 8.5,corners
7,Australia vs Turkey,cards over 2.5,cards


## Decision traces — one candidate, every number, in order

In [7]:
trace_targets = []
for family in ("advancement", "singles"):
    if stages[family]["actionable"]:
        trace_targets.append((f"{family} (actionable)",
                              stages[family]["actionable"][0]))
    if stages[family]["withheld"]:
        trace_targets.append((f"{family} (withheld)",
                              stages[family]["withheld"][0]))
for label, cand in trace_targets[:3]:
    print(f"═══ {label} ═══")
    display(pp.decision_trace(cand).to_pandas())

═══ advancement (actionable) ═══


,step,field,value
0,fixture/selection,selection,reach_QF
1,fixture/selection,market,advancement
2,fixture/selection,stage,QF
3,fixture/selection,team,Belgium
4,model input,model_prob,0.6858
5,market input,price,1.9704
6,market input,pm_price,0.5
7,"stake (¼-Kelly, capped)",stake,161.38
8,other,id,belgium_qf_pm
9,other,group,G


═══ singles (actionable) ═══


,step,field,value
0,fixture/selection,fixture,Brazil vs Norway
1,fixture/selection,selection,home
2,fixture/selection,market,1X2
3,fixture/selection,team,Brazil
4,model input,model_prob,0.5781
5,market input,price,1.8929
6,edge,edge,0.0499
7,"stake (¼-Kelly, capped)",stake,82.8
8,other,id,brazil_vs_norway_home_1x2
9,other,kickoff,2026-07-05 20:00:00+00:00


═══ singles (withheld) ═══


,step,field,value
0,fixture/selection,fixture,Canada vs Morocco
1,fixture/selection,selection,home
2,fixture/selection,market,1X2
3,fixture/selection,team,Canada
4,model input,model_prob,0.1857
5,market input,price,6.1145
6,edge,edge,0.0222
7,"stake (¼-Kelly, capped)",stake,0.0
8,governance,withheld_reason,model_prob 19% < floor 20%
9,other,id,canada_vs_morocco_home_1x2


## Parity — notebook path vs the SHIPPED feed
Same builders + same inputs ⇒ identical recommendations. Diffs appear
only when the shipped file was built from different feed snapshots
(its `meta.generated` vs our feed ages above) — that's drift visibility,
not error.

In [8]:
shipped = feeds["bet_recs_shipped"]["data"] or {}
parity = pp.parity_check(stages, shipped)
shipped_meta = (shipped.get("meta") or {})
print(f"shipped feed generated: {shipped_meta.get('generated')}")
pd.Series({k: v for k, v in parity.items() if k != "field_diffs"}).to_frame("value")

shipped feed generated: 2026-07-04 11:56:46 UTC


,value
n_notebook,17
n_shipped,3
n_shared_ids,3
only_notebook,"[belgium_final_pm, belgium_qf_pm, belgium_sf_p..."
only_shipped,[]


In [9]:
if parity["field_diffs"]:
    display(pd.DataFrame(parity["field_diffs"]))
else:
    print("no field diffs on shared IDs — full parity on stake/action/edge/price")

no field diffs on shared IDs — full parity on stake/action/edge/price


## Persist the decision snapshot (gold)

In [10]:
import lib.plotting as plot
for name, df in frames.items():
    if df.height:
        st.save_dataset(df, "gold", f"decisions_{name}", notebook="06",
                        inputs=[v["path"] for v in feeds.values()])
plot.save_table(funnel.to_pandas(), "06_decision_funnel")
print("gold decision datasets + funnel table saved")

gold decision datasets + funnel table saved


## Findings, caveats, next steps

* The production path is fully replayable in-notebook: every gate
  (edge ≥ 2pp, staleness, PM-blind, exposure, governance caps) is visible
  with the exact rows it removed and why.
* Parity against the shipped feed is ID-exact when feed snapshots match;
  the diff table quantifies drift when they don't.
* **Caveats:** dev-ledger staleness affects pool sizes (labelled);
  fresh-feed reruns belong on the mini/CI, not here.
* **Next:** notebook 07 stress-tests these same decisions under parameter
  sweeps; notebook 08 assembles the value-area view.